# Short Review Sentiment: TF-IDF vs Sentence-BERT
ARTI 404 | Section 9MA1 | Group 3 | Dr. Mohammed Mosuily
Hashim Aljarrash, Mohammed Alammari, Ahmad Muhanna

**Question:** Which method handles short reviews (6 words or fewer) better: TF-IDF + logistic regression, or Sentence-BERT + logistic regression?

Environment: Jupyter in VS Code (Python 3). Run the cells from top to bottom.

## 1. Setup
Install the packages, import them, and print the versions.

In [17]:
# Install the packages we need (already installed ones are skipped)
%pip install -q numpy pandas scikit-learn sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [18]:
import os, re, zipfile, random, urllib.request
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # hide a harmless Windows warning
import numpy as np
import pandas as pd
import sklearn, sentence_transformers
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Fixed seed so results are the same every run
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

os.makedirs("results", exist_ok=True)   # where we save our outputs

numpy: 2.1.3
pandas: 2.2.3
scikit-learn: 1.6.1
sentence-transformers: 6.3.0


## 2. Data
Download the dataset (3,000 labelled sentences: Amazon, IMDB, Yelp). 1 = Positive, 0 = Negative.

In [19]:
DATA_DIR = "data"
URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"
FILES = {"amazon": "amazon_cells_labelled.txt",
         "imdb": "imdb_labelled.txt",
         "yelp": "yelp_labelled.txt"}
os.makedirs(DATA_DIR, exist_ok=True)

# Only download if the files are not already here
if not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in FILES.values()):
    try:
        zip_path = os.path.join(DATA_DIR, "data.zip")
        urllib.request.urlretrieve(URL, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            for member in z.namelist():
                name = os.path.basename(member)
                if name in FILES.values():
                    with z.open(member) as src, open(os.path.join(DATA_DIR, name), "wb") as dst:
                        dst.write(src.read())
        print("Downloaded.")
    except Exception as e:
        print("Download failed:", e)
        print("Download manually from https://archive.ics.uci.edu/dataset/331")
        print("and put the 3 .txt files in the 'data' folder.")
else:
    print("Data already there.")

Downloaded.


Read the three files into one table.

In [20]:
rows = []
for source, fname in FILES.items():
    with open(os.path.join(DATA_DIR, fname), encoding="utf-8-sig", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            text, _, label = line.rpartition("\t")   # each line: sentence <tab> label
            if label.strip() in ("0", "1"):
                rows.append({"text": text, "label": int(label), "source": source})

raw = pd.DataFrame(rows)
print("Rows:", len(raw))
print(raw.groupby(["source", "label"]).size().unstack())

Rows: 3000
label     0    1
source          
amazon  500  500
imdb    500  500
yelp    500  500


Clean the text: lowercase, remove extra spaces, drop empty rows.

In [21]:
def clean_text(t):
    return re.sub(r"\s+", " ", str(t).lower().strip())

df = raw.copy()
df["text"] = df["text"].map(clean_text)
df = df[df["text"] != ""].reset_index(drop=True)

print("Rows after cleaning:", len(df))
print(df["label"].value_counts().rename({1: "Positive", 0: "Negative"}))
print("Duplicate texts:", df["text"].duplicated().sum())

Rows after cleaning: 3000
label
Negative    1500
Positive    1500
Name: count, dtype: int64
Duplicate texts: 21


Split into train / dev / test (70/15/15). We split on unique texts so duplicates always stay together.

In [22]:
# One row per unique text
uniq = df.groupby("text", as_index=False).agg(label=("label", lambda s: s.mode().iloc[0]))

train_u, temp_u = train_test_split(uniq, test_size=0.30, stratify=uniq["label"], random_state=SEED)
dev_u, test_u = train_test_split(temp_u, test_size=0.50, stratify=temp_u["label"], random_state=SEED)

# Give every original row its split
split_map = {}
for name, part in [("train", train_u), ("dev", dev_u), ("test", test_u)]:
    for t in part["text"]:
        split_map[t] = name
df["split"] = df["text"].map(split_map)

train = df[df.split == "train"].reset_index(drop=True)
dev   = df[df.split == "dev"].reset_index(drop=True)
test  = df[df.split == "test"].reset_index(drop=True)

# Make sure nothing is shared between splits
assert not (set(train.text) & set(dev.text))
assert not (set(train.text) & set(test.text))
assert not (set(dev.text) & set(test.text))

print(f"train={len(train)}  dev={len(dev)}  test={len(test)}  (no overlap)")
print(pd.DataFrame({"train": train.label.value_counts(),
                    "dev": dev.label.value_counts(),
                    "test": test.label.value_counts()}).rename(index={1: "Positive", 0: "Negative"}))

train=2100  dev=451  test=449  (no overlap)
          train  dev  test
label                     
Negative   1050  225   225
Positive   1050  226   224


Mark which reviews are "short" (6 words or fewer) and check how many are in the test set.

In [23]:
SHORT_MAX = 6
for d in (df, train, dev, test):
    d["n_words"] = d["text"].str.split().str.len()

print(df["n_words"].describe())
print("\nShort reviews overall:", round((df.n_words <= SHORT_MAX).mean() * 100, 1), "%")
print("Test set size:", len(test))
print("Short reviews in test:", int((test.n_words <= SHORT_MAX).sum()))

count    3000.000000
mean       11.831667
std         7.872441
min         1.000000
25%         6.000000
50%        10.000000
75%        16.000000
max        71.000000
Name: n_words, dtype: float64

Short reviews overall: 29.2 %
Test set size: 449
Short reviews in test: 130


## 3. Method 1: TF-IDF + Logistic Regression
The traditional method. We try a few settings and pick the best one on the dev set.

In [24]:
def macro_f1(y, p):
    return f1_score(y, p, average="macro", labels=[0, 1], zero_division=0)

rows_m1 = []
best_f1, tfidf_model, best_cfg = -1, None, None

for ngram in [(1, 1), (1, 2)]:          # single words, or words + word pairs
    for C in [0.1, 1, 10, 100]:         # regularisation strength
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=ngram)),
            ("clf", LogisticRegression(C=C, max_iter=1000, random_state=SEED)),
        ])
        pipe.fit(train["text"], train["label"])
        f1 = macro_f1(dev["label"], pipe.predict(dev["text"]))
        rows_m1.append({"ngram_range": ngram, "C": C, "dev_macro_f1": round(f1, 4)})
        if f1 > best_f1:
            best_f1, tfidf_model, best_cfg = f1, pipe, (ngram, C)

print(pd.DataFrame(rows_m1).to_string(index=False))
print("\nBest:", "ngram", best_cfg[0], "| C", best_cfg[1], "| dev macro F1", round(best_f1, 4))

ngram_range     C  dev_macro_f1
     (1, 1)   0.1        0.7560
     (1, 1)   1.0        0.7960
     (1, 1)  10.0        0.7982
     (1, 1) 100.0        0.7869
     (1, 2)   0.1        0.7650
     (1, 2)   1.0        0.7671
     (1, 2)  10.0        0.8003
     (1, 2) 100.0        0.7980

Best: ngram (1, 2) | C 10 | dev macro F1 0.8003


## 4. Method 2: Sentence-BERT + Logistic Regression
The pretrained method. The model turns each review into numbers, and we train only the classifier on top. (First run downloads the model, about 90 MB.)

In [25]:
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts):
    return sbert.encode(list(texts), batch_size=64, show_progress_bar=False, normalize_embeddings=True)

X_train, X_dev, X_test = embed(train.text), embed(dev.text), embed(test.text)
print("Embedding shape:", X_train.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (2100, 384)


In [26]:
# Pick C on the dev set
rows_m2 = []
best_f1, sbert_clf, best_C = -1, None, None

for C in [0.1, 1, 10, 100]:
    clf = LogisticRegression(C=C, max_iter=1000, random_state=SEED)
    clf.fit(X_train, train["label"])
    f1 = macro_f1(dev["label"], clf.predict(X_dev))
    rows_m2.append({"C": C, "dev_macro_f1": round(f1, 4)})
    if f1 > best_f1:
        best_f1, sbert_clf, best_C = f1, clf, C

print(pd.DataFrame(rows_m2).to_string(index=False))
print("\nBest: C", best_C, "| dev macro F1", round(best_f1, 4))

    C  dev_macro_f1
  0.1        0.8780
  1.0        0.8714
 10.0        0.8559
100.0        0.8426

Best: C 0.1 | dev macro F1 0.878


## 5. Results
Both methods are scored once on the same test examples (accuracy and macro F1): full test set, short reviews, and longer reviews.

In [27]:
test = test.copy()
test["pred_tfidf"] = tfidf_model.predict(test["text"])
test["pred_sbert"] = sbert_clf.predict(X_test)
test["is_short"] = test["n_words"] <= SHORT_MAX

def score_rows(name, mask):
    sub = test[mask]
    out = []
    if len(sub) == 0:      # nothing in this group
        return out
    for method, col in [("TF-IDF + LR", "pred_tfidf"), ("SBERT + LR", "pred_sbert")]:
        out.append({"Subset": name, "n": len(sub), "Method": method,
                    "Accuracy": round(accuracy_score(sub["label"], sub[col]), 4),
                    "Macro F1": round(macro_f1(sub["label"], sub[col]), 4)})
    return out

everything = test["n_words"] >= 0
table = pd.DataFrame(score_rows("Full test", everything) +
                     score_rows("Short (<=6 words)", test["is_short"]) +
                     score_rows("Longer (>6 words)", ~test["is_short"]))
print(table.to_string(index=False))
table.to_csv("results/results_table.csv", index=False)

           Subset   n      Method  Accuracy  Macro F1
        Full test 449 TF-IDF + LR    0.8441    0.8440
        Full test 449  SBERT + LR    0.9131    0.9131
Short (<=6 words) 130 TF-IDF + LR    0.8923    0.8907
Short (<=6 words) 130  SBERT + LR    0.9231    0.9226
Longer (>6 words) 319 TF-IDF + LR    0.8245    0.8234
Longer (>6 words) 319  SBERT + LR    0.9091    0.9087


The short group is small, so differences there may be due to chance.

## 6. Error analysis
Which test reviews did each method get wrong? (1 = Positive, 0 = Negative)

In [28]:
cols = ["text", "label", "pred_tfidf", "pred_sbert", "n_words"]
err_tfidf = test[test.pred_tfidf != test.label][cols]
err_sbert = test[test.pred_sbert != test.label][cols]
both_wrong = test[(test.pred_tfidf != test.label) & (test.pred_sbert != test.label)][cols]

print("TF-IDF wrong:", len(err_tfidf), "| SBERT wrong:", len(err_sbert), "| both wrong:", len(both_wrong))

pd.set_option("display.max_colwidth", 110, "display.width", 200)
print("\nTF-IDF errors (first 15):")
display(err_tfidf.head(15))
print("SBERT errors (first 15):")
display(err_sbert.head(15))
print("Both wrong:")
display(both_wrong)

err_tfidf.to_csv("results/errors_tfidf.csv", index=False)
err_sbert.to_csv("results/errors_sbert.csv", index=False)

TF-IDF wrong: 70 | SBERT wrong: 39 | both wrong: 14

TF-IDF errors (first 15):


,text,label,pred_tfidf,pred_sbert,n_words
8,good protection and does not make phone too bulky.,1,0,1,9
24,"i put the latest os on it (v1.15g), and it now likes to slow to a crawl and lock up every once in a while.",0,1,0,25
25,"the color is even prettier than i thought it would be, and the graphics are incredibly sharp.",1,0,1,17
32,i was very excited to get this headset because i thought it was really cute.,1,0,1,15
45,i was able to do voice dialing in the car with no problem.,1,0,0,13
47,the battery is working well as a replacement for the original that came with the phone over 2 years ago.,1,0,1,20
53,this is so embarassing and also my ears hurt if i try to push the ear plug into my ear.,0,1,0,20
56,magical help.,1,0,0,2
80,"small, sleek, impressive looking, practical setup with ample storage in place.",1,0,1,11
83,it's aggravating!,0,1,0,2


SBERT errors (first 15):


,text,label,pred_tfidf,pred_sbert,n_words
16,"the camera, although rated at an impressive 1.3 megapixels, renders images that fall well below expectatio...",0,0,1,22
18,)setup couldn't have been simpler.,1,1,0,5
45,i was able to do voice dialing in the car with no problem.,1,0,0,13
48,1. long lasting battery (you don't have to recharge it as frequentyly as some of the flip phones)2.,0,0,1,18
50,"i've dropped my phone more times than i can say, even on concrete and my phone is still great (knock on wo...",1,1,0,22
56,magical help.,1,0,0,2
79,the noise shield is incrediable.,1,1,0,5
89,those phones are working just fine now.,1,1,0,7
96,"they do not last forever, but is not overly expensive to replace.easy to operate and the sound is much bet...",1,0,0,25
104,the design might be ergonomic in theory but i could not stand having these in my ear.,0,0,1,17


Both wrong:


,text,label,pred_tfidf,pred_sbert,n_words
45,i was able to do voice dialing in the car with no problem.,1,0,0,13
56,magical help.,1,0,0,2
96,"they do not last forever, but is not overly expensive to replace.easy to operate and the sound is much bet...",1,0,0,25
171,the lead man is charisma-free.,0,1,1,5
205,"the soundtrack wasn't terrible, either.",1,0,0,5
206,the last 15 minutes of movie are also not bad as well.,1,0,0,12
244,you can find better movies at youtube.,0,1,1,7
264,the only consistent thread holding the series together were the amazing performances of leni parker and an...,0,1,1,26
322,"plus, it's only 8 bucks.",1,0,0,5
380,couldn't ask for a more satisfying meal.,1,0,0,7


In [29]:
# Where the two methods give different answers
disagree = test[test.pred_tfidf != test.pred_sbert][cols]
print("Methods disagree on", len(disagree), "reviews")
display(disagree.head(15))

Methods disagree on 81 reviews


,text,label,pred_tfidf,pred_sbert,n_words
8,good protection and does not make phone too bulky.,1,0,1,9
16,"the camera, although rated at an impressive 1.3 megapixels, renders images that fall well below expectatio...",0,0,1,22
18,)setup couldn't have been simpler.,1,1,0,5
24,"i put the latest os on it (v1.15g), and it now likes to slow to a crawl and lock up every once in a while.",0,1,0,25
25,"the color is even prettier than i thought it would be, and the graphics are incredibly sharp.",1,0,1,17
32,i was very excited to get this headset because i thought it was really cute.,1,0,1,15
47,the battery is working well as a replacement for the original that came with the phone over 2 years ago.,1,0,1,20
48,1. long lasting battery (you don't have to recharge it as frequentyly as some of the flip phones)2.,0,0,1,18
50,"i've dropped my phone more times than i can say, even on concrete and my phone is still great (knock on wo...",1,1,0,22
53,this is so embarassing and also my ears hurt if i try to push the ear plug into my ear.,0,1,0,20


## 7. Our own examples
Each member wrote 2 reviews of their own (1 = Positive, 0 = Negative). These are kept apart from the test scores.

In [30]:
ORIGINAL = [
    # (member, review, label)
    ("Hashim",   "Fast delivery and the phone works great.", 1),
    ("Hashim",   "The pizza was cold and the waiter ignored us.", 0),
    ("Mohammed", "Really comfortable headphones, I use them every day.", 1),
    ("Mohammed", "The battery does not last, very disappointing.", 0),
    ("Ahmad",    "Friendly staff and a very clean place.", 1),
    ("Ahmad",    "The movie was boring and far too long.", 0),
]

orig = pd.DataFrame(ORIGINAL, columns=["member", "text", "label"])
clean = orig["text"].map(clean_text)
orig["pred_tfidf"] = tfidf_model.predict(clean)
orig["pred_sbert"] = sbert_clf.predict(embed(clean))
orig["tfidf_ok"] = orig["pred_tfidf"] == orig["label"]
orig["sbert_ok"] = orig["pred_sbert"] == orig["label"]

display(orig)
print("TF-IDF correct:", int(orig.tfidf_ok.sum()), "of", len(orig))
print("SBERT correct:", int(orig.sbert_ok.sum()), "of", len(orig))
orig.to_csv("results/original_examples.csv", index=False)

,member,text,label,pred_tfidf,pred_sbert,tfidf_ok,sbert_ok
0,Hashim,Fast delivery and the phone works great.,1,1,1,True,True
1,Hashim,The pizza was cold and the waiter ignored us.,0,0,0,True,True
2,Mohammed,"Really comfortable headphones, I use them every day.",1,1,1,True,True
3,Mohammed,"The battery does not last, very disappointing.",0,0,0,True,True
4,Ahmad,Friendly staff and a very clean place.,1,1,1,True,True
5,Ahmad,The movie was boring and far too long.,0,0,0,True,True


TF-IDF correct: 6 of 6
SBERT correct: 6 of 6


## 8. Try it
Type a new review and get a prediction from both trained models (no retraining). An empty review gives a message instead of an error.

In [31]:
LABEL_NAME = {0: "Negative", 1: "Positive"}

def predict_review(text):
    # Handle empty input
    if text is None or not str(text).strip():
        print("Please enter a review, for example: 'The food was excellent'.")
        return

    t = clean_text(text)
    p_tfidf = tfidf_model.predict_proba([t])[0]
    p_sbert = sbert_clf.predict_proba(embed([t]))[0]

    print(f'Review: "{text}"')
    print(f"  TF-IDF + LR -> {LABEL_NAME[int(p_tfidf.argmax())]} ({p_tfidf.max():.2f})")
    print(f"  SBERT + LR  -> {LABEL_NAME[int(p_sbert.argmax())]} ({p_sbert.max():.2f})")
    print()

predict_review("The food was excellent")
predict_review("The food was not good")
predict_review("Great phone, works perfectly")
predict_review("Waste of money")
predict_review("   ")    # empty input example

Review: "The food was excellent"
  TF-IDF + LR -> Positive (0.81)
  SBERT + LR  -> Positive (0.86)

Review: "The food was not good"
  TF-IDF + LR -> Negative (0.96)
  SBERT + LR  -> Negative (0.88)

Review: "Great phone, works perfectly"
  TF-IDF + LR -> Positive (1.00)
  SBERT + LR  -> Positive (0.83)

Review: "Waste of money"
  TF-IDF + LR -> Negative (0.96)
  SBERT + LR  -> Negative (0.79)

Please enter a review, for example: 'The food was excellent'.


In [32]:
# Edit this text to try another review
my_review = "The service was slow but the staff were friendly"
predict_review(my_review)

Review: "The service was slow but the staff were friendly"
  TF-IDF + LR -> Negative (0.62)
  SBERT + LR  -> Negative (0.72)

